# Task 3 — Decoding Strategies: Greedy vs. Beam Search

## 1. The Search Space Problem

At inference time, the Decoder outputs a probability distribution over the target vocabulary at each step. To find the optimal sentence, we must search for the sequence of tokens that maximizes the joint probability:

$$P(y_1, y_2, ..., y_T \mid x) = \prod_{t=1}^T P(y_t \mid y_{<t}, x)$$

If our vocabulary size is $V$ and maximum sequence length is $L$, checking all possible sequences requires searching $V^L$ paths. For a modest vocabulary ($V=10,000$) and short sequence ($L=10$), this is $10^{40}$ combinations — an impossible search.

---

## 2. Greedy Decoding

**Greedy Decoding** is the simplest approach: at each step, it selects the single token with the highest probability.

- **Advantage**: Fast. It requires exactly $L$ steps of computation.
- **Limitation — The Greedy Trap**: It is locally optimal but globally suboptimal. An early choice that looks highly probable might lead to low-probability words later, or a dead end. Once a word is chosen, greedy decoding cannot backtrack.

---

## 3. Beam Search

To avoid the greedy trap, **Beam Search** keeps a set of $k$ active candidate sequences (hypotheses) at each step, where $k$ is called the **beam width**.

- At step 1, we select the top-$k$ most probable tokens.
- At each subsequent step, we expand all $k$ paths, calculate the cumulative log-probabilities for all $k \times V$ combinations, and keep only the top-$k$ paths overall.
- This process continues until they reach the `<EOS>` token or maximum length.

---

## 4. Concrete Example Walkthrough

Let's illustrate how Greedy and Beam Search (beam width $k=2$) navigate the search tree.

Assume the target vocabulary is `["they", "we", "are", "run", "fast", "<EOS>"]`. We decode the French sentence: *"ils sont rapides"* (they are fast).

### Greedy Decoding Step-by-Step
- **Step 1**: The decoder outputs token probabilities:
  - $P(\text{"they"}) = 0.51$
  - $P(\text{"we"}) = 0.49$
  - Greedy chooses **"they"** (highest probability).
- **Step 2**: Given input `"they"`, the decoder outputs transition probabilities:
  - $P(\text{"are"} \mid \text{"they"}) = 0.40$
  - $P(\text{"run"} \mid \text{"they"}) = 0.60$
  - Greedy chooses **"run"**.
- **Step 3**: Given input `"run"`, the decoder outputs:
  - $P(\text{"fast"} \mid \text{"they", "run"}) = 0.30$
  - $P(\text{"<EOS>"} \mid \text{"they", "run"}) = 0.70$
  - Greedy chooses **"<EOS>"**.
- **Greedy Output**: *"they run"* $\rightarrow$ Joint Probability: $0.51 \times 0.60 \times 0.70 =$ **$0.2142$**

---

### Beam Search Decoding Step-by-Step (k=2)
- **Step 1**: Keep the top-2 candidates:
  1. `["they"]` (probability: 0.51)
  2. `["we"]` (probability: 0.49)
- **Step 2**: Expand both candidates ($2 \times V$ options) and compute cumulative probabilities:
  - From `["they"]`:
    - `["they", "are"]` $\rightarrow 0.51 \times 0.40 = 0.204$
    - `["they", "run"]` $\rightarrow 0.51 \times 0.60 = 0.306$
  - From `["we"]`:
    - `["we", "are"]` $\rightarrow 0.49 \times 0.90 =$ **$0.441$** *(High probability transition!)*
    - `["we", "run"]` $\rightarrow 0.49 \times 0.10 = 0.049$
  - Keep the overall top-2 paths:
    1. `["we", "are"]` (probability: 0.441)
    2. `["they", "run"]` (probability: 0.306)
- **Step 3**: Expand the top-2 active paths:
  - From `["we", "are"]`:
    - `["we", "are", "fast"]` $\rightarrow 0.441 \times 0.80 =$ **$0.3528$**
    - `["we", "are", "slow"]` $\rightarrow 0.441 \times 0.10 = 0.0441$
  - From `["they", "run"]`:
    - `["they", "run", "fast"]` $\rightarrow 0.306 \times 0.30 = 0.0918$
  - Keep the overall top-2 paths:
    1. `["we", "are", "fast"]` (probability: 0.3528)
    2. `["they", "run", "fast"]` (probability: 0.0918)
- **Beam Search Output**: *"we are fast"* $\rightarrow$ Joint Probability: **$0.3528$**

### 🔑 Conclusion from Example
Greedy got trapped by picking the locally optimal word `"they"` at Step 1, missing out on the much stronger grammatical continuation `"we are fast"` ($0.3528 > 0.2142$). Beam Search successfully kept the alternative branch alive and arrived at the globally optimal sequence.

---

## 5. Essential Beam Search Adjustments
1.  **Length Normalization**: Longer sentences multiply more probability values, meaning their scores are naturally smaller (more negative in log space). To prevent the search from always favoring extremely short sentences, scores are normalized by length:
    $$\text{Normalized Score} = \frac{\sum \log P(y_t)}{\text{length}^\alpha}$$
2.  **Repetition Penalty**: Penalyzing tokens that have been generated repeatedly to prevent infinite loops.
